# Stellantis India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.stellantis.com (Google Cloud Talent Solution API)

**Brands:** Jeep, Fiat, Citroën, Peugeot, Opel/Vauxhall, Alfa Romeo, Maserati, Ram, Dodge

**Note:** Stellantis ≠ Renault. Renault has a separate career portal.

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path
SCRIPTS_DIR = Path.home() / 'Job_Scrapers' / 'All_Scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from scraper_utils import *
from bs4 import BeautifulSoup
from datetime import datetime
import requests
LOCATION_FILTER = 'India'
print('Imports loaded. Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-02 10:29:35


In [3]:
COMPANY = 'Stellantis'
OUTPUT_DIR = get_output_dir(COMPANY)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Stellantis/Outputs/2026_04_02


In [4]:
print('=' * 60)
print('STELLANTIS INDIA JOB SCRAPER')
print('ATS: Google Cloud Talent Solution API')
print('Brands: Jeep, Fiat, Citroën, Peugeot, Opel, Alfa Romeo, etc.')
print('=' * 60)

API_URL = 'https://jobsapi-google.m-cloud.io/api/job/search'
COMPANY_ORG = 'companies/16115603-6c1b-4c45-b544-238a4e6c51b3'

session = get_session()
session.headers.update({'Accept': 'application/json', 'Content-Type': 'application/json'})

stellantis_jobs = []
seen_ids = set()
page_token = ''
page_num = 0

print('  Using Google Cloud Talent Solution API...')
while len(stellantis_jobs) < 500:
    payload = {
        'companyName': COMPANY_ORG,
        'pageSize': 20,
        'offset': page_num * 20,
        'searchText': '',
    }
    if LOCATION_FILTER:
        payload['locationFilters'] = [{'address': LOCATION_FILTER, 'distanceInMiles': 0}]
    if page_token:
        payload['pageToken'] = page_token
    try:
        resp = session.post(API_URL, json=payload, timeout=30)
        if resp.status_code != 200:
            print(f'  [ERROR] HTTP {resp.status_code}')
            break
        data = resp.json()
        matched = data.get('matchingJobs', data.get('jobs', []))
        total = data.get('totalSize', data.get('total', 0))
        page_token = data.get('nextPageToken', '')
        if page_num == 0: print(f'  Total matching jobs: {total}')
        if not matched: break
        print(f'  Page {page_num+1}: {len(matched)} jobs')
        for mj in matched:
            job = mj.get('job', mj)
            title = job.get('title', '')
            if not is_valid_job_title(title): continue
            locs = job.get('locations', job.get('derivedInfo', {}).get('locations', []))
            if isinstance(locs, list) and locs:
                loc_obj = locs[0] if isinstance(locs[0], dict) else {}
                city = loc_obj.get('latLng', {}) and loc_obj.get('latLng', {}) or ''
                city = locs[0] if isinstance(locs[0], str) else loc_obj.get('address', {}).get('administrativeArea', 'India')
            else:
                city = 'India'
            if isinstance(city, str): city = city.split(',')[0].strip()
            if LOCATION_FILTER and isinstance(city, str) and LOCATION_FILTER.lower() not in str(locs).lower(): continue
            req_id = job.get('requisitionId', job.get('name', '').split('/')[-1])
            custom = job.get('customAttributes', {})
            dept = custom.get('primary_category', {}).get('stringValues', [''])[0] if isinstance(custom.get('primary_category'), dict) else ''
            posted = job.get('postingPublishTime', job.get('postingCreateTime', ''))
            desc = html_to_text(job.get('description', ''))
            job_url = f'https://careers.stellantis.com/job/{req_id}' if req_id else ''
            if req_id not in seen_ids:
                seen_ids.add(req_id)
                stellantis_jobs.append({
                    'job_id': str(req_id), 'title': title, 'company_name': 'Stellantis',
                    'job_url': job_url, 'source_api_url': API_URL,
                    'business_unit': dept, 'raw_jd_text': desc,
                    'location_city': str(city).split(',')[0].strip() if city else 'India',
                    'location_country': 'India',
                    'industry': 'Automotive / Manufacturing',
                    'date_posted': posted[:10] if posted else datetime.now().strftime('%Y-%m-%d'),
                    'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Google Cloud Talent'
                })
        if not page_token: break
        page_num += 1
        time.sleep(random.uniform(0.5, 1.5))
    except Exception as e:
        print(f'  [ERROR] {e}'); break

# Selenium fallback
if len(stellantis_jobs) < 3:
    print('\n  API returned few results. Trying Selenium on careers.stellantis.com...')
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    driver = setup_selenium()
    try:
        search_url = 'https://careers.stellantis.com/search/jobs'
        if LOCATION_FILTER: search_url += f'?location={LOCATION_FILTER}'
        driver.get(search_url); time.sleep(8)
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'a[href*="/job/"],[class*="job-result"],[class*="job-card"]')))
        except: time.sleep(5)
        for page in range(20):
            soup = BeautifulSoup(driver.page_source, 'lxml')
            cards = (soup.select('[class*="jobResultItem"]') or soup.select('[class*="job-result"]') or
                     soup.select('tr.data-row'))
            if not cards:
                links = soup.select('a[href*="/job/"]')
                seen = set()
                for link in links:
                    p = link.parent
                    if p and id(p) not in seen: cards.append(p); seen.add(id(p))
            new_count = 0
            for card in cards:
                title_el = card.select_one('[class*="title"] a') or card.select_one('a[href*="/job/"]') or card.select_one('a')
                title = title_el.get_text(strip=True) if title_el else ''
                if not is_valid_job_title(title): continue
                href = title_el.get('href', '') if title_el else ''
                job_url = href if href.startswith('http') else ('https://careers.stellantis.com' + href if href else '')
                job_id = href.rstrip('/').split('/')[-1] if href else str(abs(hash(title)))
                loc_el = card.select_one('[class*="location"]')
                loc = loc_el.get_text(strip=True) if loc_el else 'India'
                if LOCATION_FILTER and LOCATION_FILTER.lower() not in loc.lower(): continue
                if job_id not in seen_ids:
                    seen_ids.add(job_id)
                    stellantis_jobs.append({
                        'job_id': job_id, 'title': title, 'company_name': 'Stellantis',
                        'job_url': job_url, 'source_api_url': search_url, 'business_unit': '',
                        'raw_jd_text': card.get_text(' ', strip=True),
                        'location_city': loc.split(',')[0].strip(), 'location_country': 'India',
                        'industry': 'Automotive / Manufacturing',
                        'date_posted': datetime.now().strftime('%Y-%m-%d'),
                        'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Stellantis Selenium'
                    })
                    new_count += 1
            print(f'  Page {page+1}: {new_count} new jobs (total: {len(stellantis_jobs)})')
            if new_count == 0 and page > 0: break
            try:
                btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label*='Next'],a[class*='next'],[class*='pagination'] a[href*='startrow']")
                driver.execute_script('arguments[0].click();', btn); time.sleep(3)
            except: break
    except Exception as e:
        print(f'  Selenium error: {e}')
    finally:
        driver.quit()

print(f'\nTotal Stellantis India jobs: {len(stellantis_jobs)}')

STELLANTIS INDIA JOB SCRAPER
ATS: Google Cloud Talent Solution API
Brands: Jeep, Fiat, Citroën, Peugeot, Opel, Alfa Romeo, etc.
  Using Google Cloud Talent Solution API...


  [ERROR] HTTP 404

  API returned few results. Trying Selenium on careers.stellantis.com...


  Page 1: 0 new jobs (total: 0)

Total Stellantis India jobs: 0


In [5]:
df_stellantis = save_results(stellantis_jobs, 'Stellantis', OUTPUT_DIR)
if df_stellantis is not None:
    cols = ['title','location_city','seniority_level','business_unit','job_url']
    cols = [c for c in cols if c in df_stellantis.columns]
    print(df_stellantis[cols].head(10).to_string())

  [WARN] No jobs found for Stellantis
